In [21]:
import pandas as pd
import numpy as np

In this notebook, we will prepare our raw merge 2 file for regression performance. We start by importing the file as is.

In [22]:
merge2 = pd.read_csv("csv_data/MERGE2.csv")

# let's see what columns we have
merge2.columns

Index(['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'sic', 'datadate',
       'gvkey', 'conm', 'tic', 'fyear', 'at', 'ceq', 'dltt', 'lse', 'ni',
       'revt', 'xrd', 'csho', 'prcc_f', 'sich', 'mkt_cap', 'industry',
       'boardid', 'companyid', 'datestartrole', 'directorid', 'directorname',
       'companyname', 'rolename', 'dateendrole', 'datestartroleflag',
       'dateendroleflag', 'seniority', 'UG', 'top20_ug', 'MBA', 'top20_mba',
       'PhD', 'top20_phd', 'MD', 'top20_md', 'Master's', 'top20_masters',
       'dob', 'gender', 'diversitynetworklabel'],
      dtype='object')

In [23]:
merge2.isna().sum()

costat                     0
curcd                      0
datafmt                    0
indfmt                     0
consol                     0
sic                        0
datadate                   0
gvkey                      0
conm                       0
tic                        0
fyear                      0
at                         0
ceq                        0
dltt                       0
lse                        0
ni                         0
revt                       0
xrd                       87
csho                       0
prcc_f                     0
sich                       3
mkt_cap                    0
industry                   0
boardid                    0
companyid                  0
datestartrole              0
directorid                 0
directorname               0
companyname                0
rolename                   0
dateendrole                0
datestartroleflag          0
dateendroleflag            0
seniority                  0
UG            

We need to get some additional data from compustat. We get our tickers for this pull below:

In [24]:
%run processes/ticker_txt_conversion.py

We gather retained earnings from Compustat and bring them in:

In [25]:
retained_earnings = pd.read_csv("csv_data/retained_earnings.csv")

In [26]:
retained_earnings['fyear'] = pd.to_datetime(retained_earnings['datadate']).dt.year

merge2 = merge2.merge(
    retained_earnings[['gvkey', 'fyear', 're']],
    on=['gvkey', 'fyear'],
    how='left'
)

print(merge2.shape)
print(merge2['re'].isna().sum())

(1032, 48)
0


Now we are going to compute volatility:

In [28]:
volatility = pd.read_csv("csv_data/volatility.csv")
volatility['date'] = pd.to_datetime(volatility['DlyCalDt'])
volatility['fyear'] = volatility['date'].dt.year

vol_annual = volatility.groupby(['Ticker', 'fyear'])['DlyRet'].std().reset_index()
vol_annual.columns = ['tic', 'fyear', 'volatility']

merge2 = merge2.merge(
    vol_annual[['tic', 'fyear', 'volatility']],
    on=['tic', 'fyear'],
    how='left'
)

print(merge2.shape)
print(merge2['volatility'].isna().sum())

(1032, 49)
35


In [29]:
baseline_reg = merge2.copy()

In [30]:
# DEPENDENT VARIABLE
baseline_reg["roa"] = baseline_reg["ni"] / baseline_reg["at"]
baseline_reg['adjusted_roa'] = baseline_reg['roa'] - baseline_reg.groupby('fyear')['roa'].transform('mean')

# KING ET AL relevant controlls
baseline_reg['firm_size'] = np.log(baseline_reg['at'])
baseline_reg["equity_capital"] = baseline_reg["ceq"] / baseline_reg["at"]
charter_ratio = baseline_reg['mkt_cap'] / baseline_reg['ceq']
baseline_reg['charter_value'] = np.where(charter_ratio > 0, np.log(charter_ratio), np.nan)# volatility
baseline_reg["retained_earnings"] = baseline_reg["re"] / baseline_reg["at"]
baseline_reg["volatility"] = baseline_reg["volatility"]
# macro conditions

# my added controls:
# R&D
baseline_reg['xrd'] = baseline_reg['xrd'].fillna(0)
baseline_reg['rd_intensity'] = baseline_reg['xrd'] / baseline_reg['at']
#age and gender
baseline_reg["ceo_age"] = baseline_reg["fyear"] - pd.to_datetime(baseline_reg['dob']).dt.year
baseline_reg['ceo_gender'] = (baseline_reg['gender'] == 'Female').astype(int)

print(baseline_reg['charter_value'].isna().sum())


40


/Users/thefleok/Desktop/GitHub/QMSS_CEO_Thesis/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [32]:
baseline_reg.isna().sum()

costat                     0
curcd                      0
datafmt                    0
indfmt                     0
consol                     0
sic                        0
datadate                   0
gvkey                      0
conm                       0
tic                        0
fyear                      0
at                         0
ceq                        0
dltt                       0
lse                        0
ni                         0
revt                       0
xrd                        0
csho                       0
prcc_f                     0
sich                       3
mkt_cap                    0
industry                   0
boardid                    0
companyid                  0
datestartrole              0
directorid                 0
directorname               0
companyname                0
rolename                   0
dateendrole                0
datestartroleflag          0
dateendroleflag            0
seniority                  0
UG            

Let's construct our core baseline regression data by performing factor analysis.

In [36]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
import pandas as pd
import numpy as np

# All 10 education variables
edu_vars = ['UG', 'top20_ug', 
            'MBA', 'top20_mba', 
            'PhD', 'top20_phd',
            'MD', 'top20_md',
            "Master's", 'top20_masters']

# Fill any remaining NAs
baseline_reg[edu_vars] = baseline_reg[edu_vars].fillna(0)

# Standardize — factor analysis requires this
scaler = StandardScaler()
edu_scaled = scaler.fit_transform(baseline_reg[edu_vars])

# 5 factors — one per degree type like King et al.
fa = FactorAnalysis(n_components=5, random_state=42, rotation='varimax')
factors = fa.fit_transform(edu_scaled)

# Check factor loadings to verify they make sense
loadings = pd.DataFrame(
    fa.components_.T,
    index=edu_vars,
    columns=['Factor1', 'Factor2', 'Factor3', 'Factor4', 'Factor5']
)
print(loadings.round(3))

               Factor1  Factor2  Factor3  Factor4  Factor5
UG               0.047   -0.271    0.029    0.109    0.416
top20_ug         0.018    0.008    0.127    0.116    0.476
MBA             -0.071   -0.070   -0.082    0.751   -0.024
top20_mba       -0.026   -0.052   -0.015    0.884    0.040
PhD              0.153    0.201    0.549   -0.083   -0.272
top20_phd        0.131    0.024    0.944   -0.032    0.086
MD              -0.050    0.958    0.042   -0.066   -0.028
top20_md         0.104    0.372    0.235   -0.011    0.197
Master's         0.955   -0.046    0.104   -0.029   -0.015
top20_masters    0.559    0.002    0.337   -0.124    0.430


Factor 1 is heavily connected to the master's degree (0.955 and 0559). Factor 2 is heavily connected to the MD (0.958 and 0.372). Factor 3 is heavily connected to the top20 phd (0.944 and 0.549 for phd). Factor 4 is heavily connected to the MBA (0.751 and 0.884). Factor 5 is heavily connected to undergrad (0.416 and 0.476).

In [37]:
baseline_reg['masters_factor'] = factors[:, 0]
baseline_reg['md_factor'] = factors[:, 1]
baseline_reg['phd_factor'] = factors[:, 2]
baseline_reg['mba_factor'] = factors[:, 3]
baseline_reg['ug_factor'] = factors[:, 4]

print(baseline_reg[['ug_factor', 'mba_factor', 'phd_factor', 
                     'md_factor', 'masters_factor']].describe())

          ug_factor    mba_factor    phd_factor     md_factor  masters_factor
count  1.032000e+03  1.032000e+03  1.032000e+03  1.032000e+03    1.032000e+03
mean  -1.377021e-17 -5.852338e-17  9.983401e-17 -4.131062e-17    1.377021e-17
std    7.361749e-01  9.127840e-01  9.553689e-01  9.623935e-01    9.615885e-01
min   -1.766358e+00 -6.535556e-01 -5.980047e-01 -5.116850e-01   -1.184203e+00
25%   -5.106751e-01 -6.182375e-01 -2.491388e-01 -2.753060e-01   -6.773651e-01
50%    8.550763e-02 -4.961084e-01 -1.880647e-01 -2.128972e-01   -6.128737e-01
75%    1.164895e-01  4.526848e-02 -1.428900e-01 -1.334297e-01    1.328120e+00
max    2.673622e+00  1.832359e+00  4.350235e+00  5.008076e+00    1.640976e+00


In [39]:
baseline_reg.isna().sum()

costat            0
curcd             0
datafmt           0
indfmt            0
consol            0
                 ..
masters_factor    0
md_factor         0
phd_factor        0
mba_factor        0
ug_factor         0
Length: 63, dtype: int64

In [40]:
BASELINE1 = baseline_reg.to_csv("csv_data/BASELINE1.csv", index=False)